In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
import joblib

# 1. Generate a Built-In "Synthetic" Customer Churn Dataset
# We create 10,000 customers with 10 features (simulating Age, Balance, Credit Score, etc.)
print("Generating synthetic bank customer data...")
X_raw, y = make_classification(
    n_samples=10000,
    n_features=10,
    n_informative=8,  # 8 features actually affect churn
    n_redundant=2,    # 2 features are just noise/correlated
    weights=[0.8, 0.2], # Simulates an 80% stay / 20% churn class imbalance
    random_state=42
)

# Optional: Put it in a Pandas DataFrame just so you can see what it looks like
feature_names = [f"Feature_{i}" for i in range(1, 11)]
df = pd.DataFrame(X_raw, columns=feature_names)
df['Churn'] = y
print("Dataset Shape:", df.shape)
print("Churn Distribution:\n", df['Churn'].value_counts())

# 2. Train-Test Split (80% Training, 20% Testing)
X = df.drop('Churn', axis=1).values
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Scale the Data (Essential for Keras Models)
print("Scaling features...")
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 4. Build the Deep Learning Model using Keras
print("Building Keras model...")
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train_scaled.shape[1],)),
    Dropout(0.3), # Drops 30% of connections randomly to prevent memorization
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid') # Outputs the final churn probability (0.0 to 1.0)
])

# Compile the model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 5. Train the Model
# EarlyStopping prevents the model from training too long and overfitting
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

print("Training model...")
history = model.fit(
    X_train_scaled, y_train,
    epochs=50,
    batch_size=32,
    validation_data=(X_test_scaled, y_test),
    callbacks=[early_stop],
    verbose=1
)

# 6. Evaluate and Export
loss, accuracy = model.evaluate(X_test_scaled, y_test)
print(f"\n✅ Final Model Accuracy on Unseen Data: {accuracy*100:.2f}%")

# Save the trained model and scaler so you can use them in your MERN/FastAPI backend
model.save("synthetic_churn_model.keras")
joblib.dump(scaler, "synthetic_scaler.pkl")
print("✅ Artifacts saved as 'synthetic_churn_model.keras' and 'synthetic_scaler.pkl'")

Generating synthetic bank customer data...
Dataset Shape: (10000, 11)
Churn Distribution:
 Churn
0    7968
1    2032
Name: count, dtype: int64
Scaling features...
Building Keras model...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:93: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Training model...
Epoch 1/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - accuracy: 0.7570 - loss: 0.4977 - val_accuracy: 0.8840 - val_loss: 0.2875
Epoch 2/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.8687 - loss: 0.2981 - val_accuracy: 0.9255 - val_loss: 0.2225
Epoch 3/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9063 - loss: 0.2374 - val_accuracy: 0.9380 - val_loss: 0.1954
Epoch 4/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9146 - loss: 0.2157 - val_accuracy: 0.9390 - val_loss: 0.1760
Epoch 5/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9227 - loss: 0.2050 - val_accuracy: 0.9425 - val_loss: 0.1690
Epoch 6/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9288 - loss: 0.1880 - val_accuracy: 0.9460 - val_loss: 0.1570
Epoch 7/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.9382 - loss: 0.1803 - val_accuracy: 0.9505 - val_loss: 0.1499
Epoch 8/50
250/250 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9479 - loss: 0.1578 

In [ ]:
import os

os.makedirs('public', exist_ok=True)

server_code = """
from fastapi import FastAPI, HTTPException
from fastapi.staticfiles import StaticFiles
from pydantic import BaseModel
import numpy as np
import joblib
from tensorflow.keras.models import load_model

app = FastAPI()

model = load_model('synthetic_churn_model.keras')
scaler = joblib.load('synthetic_scaler.pkl')

# Feature names to make the explanation human-readable
FEATURE_NAMES = ["Age", "Balance", "Credit Score", "Tenure", "Num Products",
                 "Has Credit Card", "Is Active Member", "Estimated Salary", "Complaints", "Satisfaction"]

# --- MOCK DATABASE ---
USERS = {
    "alice": {"password": "password123", "role": "customer", "name": "Alice Smith",
              "features": [42, 50000, 650, 2, 1, 1, 1, 80000, 0, 4]},
    "bob": {"password": "password123", "role": "customer", "name": "Bob Jones",
            "features": [25, 100, 580, 1, 1, 0, 0, 30000, 2, 2]},
    "admin": {"password": "admin", "role": "admin", "name": "Super Admin", "features": []}
}

class LoginData(BaseModel):
    username: str
    password: str

# 1. Login Endpoint
@app.post("/api/login")
def login(data: LoginData):
    user = USERS.get(data.username.lower())
    if not user or user["password"] != data.password:
        raise HTTPException(status_code=401, detail="Invalid credentials")
    return {"username": data.username, "role": user["role"], "name": user["name"]}

# 2. Get Account Details
@app.get("/api/account/{username}")
def get_account(username: str):
    user = USERS.get(username)
    if not user or user["role"] != "customer":
        raise HTTPException(status_code=404, detail="Customer not found")
    # Zip the feature names with the raw values for the UI
    details = [{"name": n, "value": v} for n, v in zip(FEATURE_NAMES, user["features"])]
    return {"name": user["name"], "details": details}

# 3. Advanced Prediction Endpoint (With Explainability)
@app.get("/api/predict/{username}")
def predict_churn(username: str):
    user = USERS.get(username)
    raw_features = user["features"]

    # Scale the features
    input_scaled = scaler.transform([raw_features])[0]

    # Predict
    probability = float(model.predict(np.array([input_scaled]))[0][0])

    # Explainability: Which features are pulling the model the most?
    # Features furthest from 0.0 (the scaled mean) are the most unusual for this customer
    explainability = []
    for name, raw, scaled in zip(FEATURE_NAMES, raw_features, input_scaled):
        explainability.append({
            "feature": name,
            "raw_value": raw,
            "scaled_weight": round(scaled, 2),
            "impact": "High" if abs(scaled) > 1.0 else "Normal"
        })

    return {
        "churn_probability": probability,
        "is_high_risk": probability > 0.5,
        "explanation": explainability
    }

# 4. Admin Endpoint: Get All Customers
@app.get("/api/admin/customers")
def get_all_customers():
    return [{"username": k, "name": v["name"]} for k, v in USERS.items() if v["role"] == "customer"]

app.mount("/", StaticFiles(directory="public", html=True), name="public")
"""

with open('server.py', 'w') as f:
    f.write(server_code)
print("✅ Advanced Server Created!")

✅ Advanced Server Created!


In [ ]:
from google.colab import files

files.download('synthetic_churn_model.keras')
files.download('synthetic_scaler.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import urllib

!nohup uvicorn server:app --port 8000 &

ip_address = urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n")
print(f"🔑 YOUR TUNNEL PASSWORD IS: {ip_address}")
print("Click the link below and enter the password.")

!npx localtunnel --port 8000

nohup: appending output to 'nohup.out'
🔑 YOUR TUNNEL PASSWORD IS: 34.11.62.195
Click the link below and enter the password.
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦your url is: https://clear-sites-attack.loca.lt
^C
